In [50]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor 

from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

In [51]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
df = pd.read_csv('Cleaned_Merged_Industry_Jobs.csv')
df.head(2)

,Job_ID,Company,Role,Role_Group,Role_Group_Cloud Engineer,Role_Group_DevOps Engineer,Role_Group_Network Engineer,Role_Group_Other,Role_Group_Security/SOC Analyst,Role_Group_Site Reliability Engineer,Experience_Encoded,Experience_Unknown,Company_Freq,PythonRequired,LinuxRequired,NetworkingRequired,AWSRequired,AzureRequired,DockerRequired,KubernetesRequired,TerraformRequired,CyberSecurityRequired,CommunicationRequired,PythonRequired_scaled,LinuxRequired_scaled,NetworkingRequired_scaled,AWSRequired_scaled,AzureRequired_scaled,DockerRequired_scaled,KubernetesRequired_scaled,TerraformRequired_scaled,CyberSecurityRequired_scaled,CommunicationRequired_scaled,SalaryLPA_parsed,SalaryLPA_parsed_scaled
0,1,Capgemini,Cloud Security Engineer,Security/SOC Analyst,0,0,0,0,1,0,0.0,0,0.030488,5,9,8,9,6,7,9,5,8,8,-0.459834,0.809277,0.671253,0.841033,0.139885,0.334558,0.973696,-0.030849,1.302136,0.662521,9.61,2.222266
1,2,HCL,Cloud Engineer,Cloud Engineer,1,0,0,0,0,0,0.0,0,0.012195,7,10,8,7,9,9,5,7,7,8,0.477985,1.483675,0.671253,0.203575,1.110732,1.008019,-0.297234,0.652239,0.941078,0.662521,5.42,0.238335


In [52]:
df.columns

Index(['Job_ID', 'Company', 'Role', 'Role_Group', 'Role_Group_Cloud Engineer',
       'Role_Group_DevOps Engineer', 'Role_Group_Network Engineer',
       'Role_Group_Other', 'Role_Group_Security/SOC Analyst',
       'Role_Group_Site Reliability Engineer', 'Experience_Encoded',
       'Experience_Unknown', 'Company_Freq', 'PythonRequired', 'LinuxRequired',
       'NetworkingRequired', 'AWSRequired', 'AzureRequired', 'DockerRequired',
       'KubernetesRequired', 'TerraformRequired', 'CyberSecurityRequired',
       'CommunicationRequired', 'PythonRequired_scaled',
       'LinuxRequired_scaled', 'NetworkingRequired_scaled',
       'AWSRequired_scaled', 'AzureRequired_scaled', 'DockerRequired_scaled',
       'KubernetesRequired_scaled', 'TerraformRequired_scaled',
       'CyberSecurityRequired_scaled', 'CommunicationRequired_scaled',
       'SalaryLPA_parsed', 'SalaryLPA_parsed_scaled'],
      dtype='str')

In [53]:
df.drop(columns=['Job_ID','Company','Role','Role_Group','Experience_Unknown','PythonRequired', 'LinuxRequired',
       'NetworkingRequired', 'AWSRequired', 'AzureRequired', 'DockerRequired',
       'KubernetesRequired', 'TerraformRequired', 'CyberSecurityRequired',
       'CommunicationRequired','SalaryLPA_parsed'], inplace=True)

In [54]:
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [55]:
linear_model = LinearRegression()

In [56]:
model_rf = RandomForestRegressor(n_estimators=200, random_state=42,max_depth=None,)

In [57]:
model_svr = SVR(kernel='rbf', C=1.0, epsilon=0.1)

In [58]:
model_xg = XGBRegressor( n_estimators=100, 
                     learning_rate=0.05, 
                     subsample=0.8, 
                     colsample_bytree=0.8, 
                     random_state=42 )

In [59]:
models = {
    "Linear Regression": linear_model,
    "Random Forest": model_rf,
    "SVR": model_svr,
    "XGBoost": model_xg
}

In [60]:
scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

results = []

In [61]:
for name, model in models.items():

    scores = cross_validate(
        model,
        x,
        y,
        cv=kf,
        scoring=scoring,
        n_jobs=-1
    )

    mae_scores = -scores["test_MAE"]
    rmse_scores = -scores["test_RMSE"]
    r2_scores = scores["test_R2"]

    results.append({
        "Model": name,
        "MAE Mean": mae_scores.mean(),
        "RMSE Mean": rmse_scores.mean(),
        "R2 Mean": r2_scores.mean(),
    })

In [62]:
results_df = pd.DataFrame(results)

In [63]:
display(results_df.round(4))

,Model,MAE Mean,RMSE Mean,R2 Mean
0,Linear Regression,0.6827,0.8745,0.1477
1,Random Forest,0.4053,0.5867,0.5834
2,SVR,0.5436,0.7613,0.3610
3,XGBoost,0.3710,0.5541,0.6019


### Conclusion

The 5-fold cross-validation results show that **XGBoost is the best-performing model** among the four evaluated models. It achieved the lowest MAE (0.3710) and RMSE (0.5541), while also obtaining the highest R² score (0.6019). This indicates that XGBoost provides the most accurate predictions and explains approximately **60.2% of the variation in the target variable**. Random Forest was the second-best model, while SVR and Linear Regression showed comparatively weaker performance. Therefore, **XGBoost was selected as the final model** for the prediction task.